# Packing the `data` directory on Google Drive into `data.tar`

This notebook runs on Colab: it packs the group's `data` directory on Drive into one `data.tar` file and places it next to the directory in `DRIVE_PROJECT_DIR`, exactly where `parser.ipynb` looks for it (`data.tar.gz` → `data.tgz` → `data.tar`).

Steps:
1. Mount Drive, check the source directory, count files and size.
2. Create the tar file on the runtime's **local disk** (`/content`), because writing one large file to Drive through FUSE is slow and can break midway.
3. Check the tar file (count entries and compare with the number of source files).
4. Copy it to Drive under a temporary `*.partial` name, then rename it, so `parser.ipynb` never sees a half-copied file.

The tar file holds a `data/` directory at its root (like the existing `data.tar`), so it extracts to `.../data/<paper_id>/...`.

**Note:** the tar file is a snapshot taken at creation time; rerun this notebook when the data on Drive changes.

## 1. Configuration

In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intro2DS-Group3"  # directory that contains `data/`
SOURCE_NAME = "data"            # name of the directory to pack, inside DRIVE_PROJECT_DIR
COMPRESS = False                # False -> data.tar; True -> data.tar.gz (smaller but slower to pack/extract)
EXCLUDE = []                    # patterns to skip, e.g. [".ipynb_checkpoints", "*.tmp"]
LOCAL_WORK_DIR = "/content"     # runtime local disk, where the tar is created before copying to Drive
OVERWRITE = True                # False: stop if the target file already exists on Drive

## 2. Mount Drive and check the source

In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running on Colab: skipping the Drive mount.")

project_dir = Path(DRIVE_PROJECT_DIR)
source_dir = project_dir / SOURCE_NAME
if not source_dir.is_dir():
    raise FileNotFoundError(f"Directory {source_dir} not found. Check DRIVE_PROJECT_DIR / the Drive shortcut.")

archive_name = SOURCE_NAME + (".tar.gz" if COMPRESS else ".tar")
local_archive = Path(LOCAL_WORK_DIR) / archive_name
drive_archive = project_dir / archive_name

if drive_archive.exists() and not OVERWRITE:
    raise FileExistsError(f"{drive_archive} already exists; set OVERWRITE = True to replace it.")

# parser.ipynb prefers data.tar.gz > data.tgz > data.tar, so an old archive with another extension hides the new one.
for other in ("data.tar.gz", "data.tgz", "data.tar"):
    p = project_dir / other
    if other != archive_name and p.is_file():
        print(f"WARNING: {p} already exists. parser.ipynb may use that file instead of {archive_name}; "
              "delete or rename it if it is outdated.")

print(f"Counting files in {source_dir} (this can take a few minutes on Drive)...")
started = time.perf_counter()
n_files = n_dirs = total_bytes = 0
for root, dirs, files in os.walk(source_dir):
    n_dirs += len(dirs)
    for name in files:
        n_files += 1
        try:
            total_bytes += os.lstat(os.path.join(root, name)).st_size
        except OSError:
            pass
print(f"{n_files:,} files, {n_dirs:,} subdirectories, {total_bytes / 2**30:.2f} GiB "
      f"(counted in {time.perf_counter() - started:.0f}s)")

free = shutil.disk_usage(LOCAL_WORK_DIR).free
print(f"Free local disk space: {free / 2**30:.2f} GiB")
# tar adds a 512-byte header per entry and rounds to 512-byte blocks.
needed = total_bytes + (n_files + n_dirs) * 1024 + 2**26
if free < needed:
    raise RuntimeError(f"Not enough local disk space: about {needed / 2**30:.2f} GiB needed.")

## 3. Create the tar file on the local disk

In [ ]:
local_archive.unlink(missing_ok=True)

cmd = ["tar", "-c", "-f", str(local_archive), "-C", str(project_dir)]
if COMPRESS:
    # pigz compresses in parallel, much faster than gzip; fall back to gzip if the runtime has no pigz.
    cmd += ["--use-compress-program=pigz"] if shutil.which("pigz") else ["-z"]
for pattern in EXCLUDE:
    cmd += [f"--exclude={pattern}"]
cmd += [SOURCE_NAME]
print(" ".join(cmd))

log_path = Path(LOCAL_WORK_DIR) / (archive_name + ".log")
started = time.perf_counter()
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    last_report = started
    while proc.poll() is None:
        time.sleep(1)
        if time.perf_counter() - last_report < 15:   # report progress every 15s
            continue
        last_report = time.perf_counter()
        done = local_archive.stat().st_size if local_archive.exists() else 0
        pct = f" (~{min(99, 100 * done / total_bytes):.0f}%)" if total_bytes and not COMPRESS else ""
        print(f"[{last_report - started:6.0f}s] {done / 2**30:.2f} GiB{pct}")
messages = log_path.read_text(errors="replace").strip()
if messages:
    print("tar messages:\n" + messages[-3000:])
if proc.returncode != 0:
    raise RuntimeError(f"tar failed (exit {proc.returncode}); see {log_path}.")

tar_seconds = time.perf_counter() - started
size = local_archive.stat().st_size
print(f"Created {local_archive}: {size / 2**30:.2f} GiB in {tar_seconds:.0f}s")

## 4. Check the tar file

In [ ]:
listing = subprocess.run(["tar", "-tf", str(local_archive)], capture_output=True, text=True, check=True).stdout.splitlines()
n_entries = len(listing)
n_tar_files = sum(1 for e in listing if not e.endswith("/"))
print(f"The tar has {n_entries:,} entries ({n_tar_files:,} files). Source: {n_files:,} files.")
print("First entries:", *listing[:5], sep="\n  ")

if not all(e == SOURCE_NAME + "/" or e.startswith(SOURCE_NAME + "/") for e in listing):
    raise RuntimeError("The tar has entries outside the expected root directory.")
if not EXCLUDE and n_tar_files != n_files:
    raise RuntimeError("The tar file count differs from the source; the data may have changed while packing. Rerun sections 2–4.")
print("OK")

## 5. Copy to Drive

In [ ]:
partial = drive_archive.with_name(drive_archive.name + ".partial")
partial.unlink(missing_ok=True)
print(f"Copying {local_archive} → {drive_archive} ...")
started = time.perf_counter()
shutil.copyfile(local_archive, partial)
if partial.stat().st_size != local_archive.stat().st_size:
    partial.unlink(missing_ok=True)
    raise RuntimeError("The size of the copy on Drive does not match; try running this cell again.")
os.replace(partial, drive_archive)
print(f"Done after {time.perf_counter() - started:.0f}s: {drive_archive} ({drive_archive.stat().st_size / 2**30:.2f} GiB)")
print("Drive syncs in the background; wait a few minutes before closing the runtime so the file is fully uploaded.")

## 6. (Optional) Clean up the local disk or download

- Delete the local copy to free space: run the cell below.
- Download to your machine: for large files (several GiB), downloading from Google Drive in the browser is more reliable than `files.download`.

In [ ]:
# local_archive.unlink(missing_ok=True)

# from google.colab import files
# files.download(str(local_archive))